# Выбор рабочих веток тяжёлых и лёгких цепей мыши

Ноутбук использует завершённые сопоставимые сравнения и записывает явные манифесты выбранных веток. Выбор основан на абсолютном числе строгих продуктивных полных последовательностей без стоп-кодона от V до J; для эмпирического удаления Cκ дополнительно проверяются не меньшая результативность и остаточные мотивы.


In [ ]:
from pathlib import Path
import hashlib,json,os
START=Path.cwd().resolve();REPO=next((p for p in (START,*START.parents) if (p/'results').is_dir()),None)
if REPO is None:raise RuntimeError(f'cannot locate repo from {START}')
VOLUME=Path(os.environ.get('BCR_VOLUME','/data/user/epishkin'))
if not (VOLUME/'results').is_dir():VOLUME=REPO
RESULTS=VOLUME/'results';SAMPLES=['ERR346596','ERR346597','ERR346598','ERR346599','ERR346600','ERR346601']
def sha256(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(2**20),b''):h.update(b)
 return h.hexdigest()
def checked(paths):
 for path in paths.values():
  assert path.is_file() and path.stat().st_size>0,path
 return {k:str(v.resolve()) for k,v in paths.items()}


In [ ]:
# Выбор ветки тяжёлых цепей.
heavy_report_path=RESULTS/'ERP003950_fastp_q30_u40/heavy_fastp_comparison.json';heavy_report=json.loads(heavy_report_path.read_text())
heavy_label=max(heavy_report,key=lambda x:heavy_report[x]['totals']['strict_source_records'])
heavy_root={'legacy_q30_trim':RESULTS/'ERP003950','fastp_q30_u40':RESULTS/'ERP003950_fastp_q30_u40'}[heavy_label]
heavy_manifests={s:heavy_root/'comparison_annotation/strict_truth'/s/f'{s}_{heavy_label}_template_manifest.tsv' for s in SAMPLES}
heavy_fastas={s:heavy_root/'comparison_annotation/strict_truth'/s/f'{s}_{heavy_label}_unique.fasta' for s in SAMPLES}
heavy_ref_root=RESULTS/'PRJNA1226555/references/germline'
heavy_refs={'V':heavy_ref_root/'ncbi_mouse/mouse_gl_V.nsq','D':heavy_ref_root/'ncbi_mouse/mouse_gl_D.nsq','J':heavy_ref_root/'ncbi_mouse/mouse_gl_J.nsq','aux':heavy_ref_root/'igdata_balbcbyj/optional_file/mouse_gl.aux'}
heavy_filter_summary=heavy_root/'trimmed/filter_summary.json'
heavy_preprocessing={'branch_semantics':{'legacy_q30_trim':'cutadapt adapter plus 3-prime Q30 quality trimming; min length 250','fastp_q30_u40':'cutadapt adapters-only then fastp whole-read filter q30/u40/min250; no quality-end trimming'}[heavy_label]}
if heavy_filter_summary.is_file():heavy_preprocessing['filter_summary']=json.loads(heavy_filter_summary.read_text())
heavy_selection={'dataset':'ERP003950','selected_branch':heavy_label,'selected_root':str(heavy_root.resolve()),'rejected_comparators':[x for x in heavy_report if x!=heavy_label],'expected_loci':['IGH'],'strict_manifests':checked(heavy_manifests),'strict_fastas':checked(heavy_fastas),'selection_metric':'strict_source_records','selection_rationale':'maximum absolute strict productive complete no-stop IGH V-through-J source mass under identical collapse and annotation','comparison_report':str(heavy_report_path.resolve()),'metrics':heavy_report[heavy_label]['totals'],'input_output_counts':heavy_report[heavy_label]['totals'],'preprocessing_parameters':heavy_preprocessing,'annotation_reference_checksums':{k:{'path':str(v.resolve()),'sha256':sha256(v)} for k,v in heavy_refs.items()},'native_heavy_light_pairing':False}
heavy_out=RESULTS/'ERP003950_selected_branch.json';heavy_out.write_text(json.dumps(heavy_selection,indent=2)+chr(10));print(heavy_out,heavy_label)


In [ ]:
# Выбор ветки лёгких цепей: базовая минимальная длина и кандидат с удалением пяти мотивов.
primer_path=RESULTS/'PRJNA1226555/comparisons/light_primer_strategy_comparison.json';primer=json.loads(primer_path.read_text())
minlen_path=RESULTS/'PRJNA1226555/comparisons/light_fastp_minlen_comparison.json';minlen=json.loads(minlen_path.read_text())
baseline=primer['paper4_cut'];candidate=primer['paper4_plus_ck_cut']
baseline_ck=baseline['raw'].get('ck_full_exact',0);candidate_ck=candidate['raw'].get('ck_full_exact',0)
ck_reduced=(baseline_ck==0 and candidate_ck==0) or candidate_ck<=0.1*baseline_ck
use_candidate=bool(primer['candidate_passes_noninferiority'] and ck_reduced)
light_branch=primer['candidate_branch'] if use_candidate else primer['source_branch']
light_root=RESULTS/'PRJNA1226555/branches'/light_branch
light_run=f'SRR32426580_{light_branch}' if use_candidate else f"SRR32426580_{light_branch.rsplit('_',1)[-1]}"
light_truth=light_root/'annotation/strict_truth'
light_manifests={'SRR32426580':light_truth/f'{light_run}_template_manifest.tsv'}
light_fastas={'SRR32426580':light_truth/f'{light_run}_unique.fasta'}
chosen=candidate if use_candidate else baseline
light_ref_root=RESULTS/'PRJNA1226555/references/germline'
light_refs={'V':light_ref_root/'igblast_db/balbcbyj_igkv_iglv.nsq','D':light_ref_root/'ncbi_mouse/mouse_gl_D.nsq','J':light_ref_root/'igblast_db/all_strains_IGKLJ.nsq','aux':light_ref_root/'ogrdb_balbcbyj/all_strains_IGKLJ.aux'}
preprocessing_parameters={}
for rel,key in [('trimmed/trim_summary.json','trim'),('pr_trimmed/primer_summary.json','primer'),('merged/assembly_summary.json','merge')]:
 p=light_root/rel
 if p.is_file():preprocessing_parameters[key]=json.loads(p.read_text())
rejected=[x for x in [primer['source_branch'],primer['candidate_branch']] if x!=light_branch]
selected_minlen=primer['source_branch'].rsplit('_',1)[-1]
rejected.extend(f'fastp_q30_u40_{x}' for x in ('min200','min250') if x!=selected_minlen)
light_selection={'dataset':'PRJNA1226555','selected_branch':light_branch,'selected_root':str(light_root.resolve()),'rejected_comparators':rejected,'expected_loci':['IGK','IGL'],'strict_manifests':checked(light_manifests),'strict_fastas':checked(light_fastas),'selection_metric':'strict accepted_source_records with Ck noninferiority/residual gate','selection_rationale':'maximum absolute strict light-chain V-through-J yield for min-length, then adopt empirical Ck cut only with >=99% noninferiority and >=90% exact-motif reduction','comparison_reports':[str(minlen_path.resolve()),str(primer_path.resolve())],'metrics':chosen['strict'],'input_output_counts':{'selected_minlen':minlen['comparison'][selected_minlen],'selected_primer_strategy':chosen},'preprocessing_parameters':preprocessing_parameters,'annotation_reference_checksums':{k:{'path':str(v.resolve()),'sha256':sha256(v)} for k,v in light_refs.items()},'ck_candidate_adopted':use_candidate,'ck_source_status':'empirical_candidate_not_kit_source_backed','native_heavy_light_pairing':False}
light_out=RESULTS/'PRJNA1226555/selected_branch.json';light_out.write_text(json.dumps(light_selection,indent=2)+chr(10));print(light_out,light_branch)


In [ ]:
# Компактный отчёт для чтения человеком.
report=REPO/'docs/mouse_heavy_light_branch_selection_report.md'
report.write_text(chr(10).join([
 '# Mouse heavy/light branch selection','',
 f'- Heavy selected: `{heavy_label}`; strict source records: `{heavy_selection["metrics"]["strict_source_records"]}`.',
 f'- Light selected: `{light_branch}`; strict source records: `{light_selection["metrics"]["accepted_source_records"]}`.',
 f'- Empirical Cκ cut adopted: `{use_candidate}`.',
 '- Cκ motif status: empirical candidate, not a source-backed proprietary kit oligo.',
 '- Native H–L pairing: not represented.',
 '',f'- Heavy manifest: `{heavy_out}`.',f'- Light manifest: `{light_out}`.','']))
print(report)
